# M-Shots Learning


In this lab, we'll explore zero-shot, one-shot, and few-shot prompting. We'll compare how examples and explicit instructions influence the consistency and format of model responses.


## Setup

Install the dependencies if needed (`pip install -U openai python-dotenv`). Create a `.env` file containing `OPENAI_API_KEY=your_api_key`, or set the environment variable in your shell. Never commit your API key to version control.


In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

load_dotenv(find_dotenv())

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Set OPENAI_API_KEY in your environment or .env file.")

client = OpenAI()
MODEL = "gpt-4o-mini"


## Few-shot prompting

A *shot* is an example included in a prompt:

- **Zero-shot:** no examples; the model relies on the instructions and its prior training.
- **One-shot:** one example.
- **Few-shot:** multiple examples.

Examples can help a model follow a desired style, label set, or output format. There is no universally correct number of examples: results depend on the task, the examples, and the model. Examples also consume input tokens, so use a small, representative set and evaluate the results.

This notebook uses OpenAI's **Responses API** with `gpt-4o-mini`. The API accepts a list of role-based messages as `input`; the response text is available through `response.output_text`.


In [6]:
def return_OAIResponse(user_message, context):
    """Send role-based prompt context and a question to the Responses API."""
    messages = [*context, {"role": "user", "content": f"Question: {user_message}"}]
    response = client.responses.create(
        model=MODEL,
        input=messages,
    )
    return response.output_text


## 1. Zero-shot prompting

With no examples, the model may answer correctly but choose its own format. Inspect the output and note which details are present or missing.


In [7]:
context_user = [
    {"role": "system", "content": "You are an expert in Formula One. Answer factual questions clearly."}
]

In [8]:
# let's see the result with zero-shot:
print(return_OAIResponse("Who won the 2010 F1 Drivers' Championship?", context_user))

RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

## 2. One-shot prompting

Now include one example as a user/assistant exchange. The example demonstrates the format we want; it does not guarantee that every answer will be correct.


In [ ]:
context_user = [
    {"role": "system", "content": "Answer F1 championship questions using exactly these fields: Driver, Team, Points."},
    {"role": "user", "content": "Who won the 2000 F1 Drivers' Championship?"},
    {"role": "assistant", "content": "Driver: Michael Schumacher\nTeam: Ferrari\nPoints: 108"},
]

In [ ]:
# let's see the result with one-shot:
print(return_OAIResponse("Who won the 2011 F1 Drivers' Championship?", context_user))

## 3. Few-shot prompting

Multiple examples show the format across more than one case. Keep examples accurate and representative of the task.


In [ ]:
context_user = [
    {"role": "system", "content": "Answer F1 championship questions using exactly these fields: Driver, Team, Points."},
    {"role": "user", "content": "Who won the 2010 F1 Drivers' Championship?"},
    {"role": "assistant", "content": "Driver: Sebastian Vettel\nTeam: Red Bull Racing\nPoints: 256"},
    {"role": "user", "content": "Who won the 2009 F1 Drivers' Championship?"},
    {"role": "assistant", "content": "Driver: Jenson Button\nTeam: Brawn GP\nPoints: 95"},
]
print(return_OAIResponse("Who won the 2006 F1 Drivers' Championship?", context_user))


In [ ]:
# let's see the result with few-shot:
print(return_OAIResponse("Who won the 2019 F1 Drivers' Championship?", context_user))

## 4. Examples as messages versus instructions

Role-based examples make the demonstration clearer: each example question is a `user` message and its expected answer is an `assistant` message. The system message states the task. You can compare this with a prompt that explains the format directly. Neither approach is always best; test both on the same questions.


In [ ]:
context_user = [
    {"role": "system", "content": "You are an expert in Formula One. Answer with exactly these fields, each on its own line: Driver, Team, Points. Use the championship winner's season-end points."}
]
print(return_OAIResponse("Who won the 2019 F1 Drivers' Championship?", context_user))


## 5. Classification with few-shot examples

Few-shot prompting can also be useful for other types of tasks, including classification. By providing examples of inputs and their expected labels, we can guide a model toward the desired behavior.

In this example, we'll classify product reviews by sentiment into three categories: **Positive**, **Negative**, or **Neutral**. Notice how the examples illustrate both the task and the expected output format.


In [ ]:
context_user = [
    {"role": "system", "content": "Classify the product review as Positive, Negative, or Neutral. Return only the label."},
    {"role": "user", "content": "It fulfilled its function perfectly, the price is fair, and I would buy it again."},
    {"role": "assistant", "content": "Positive"},
    {"role": "user", "content": "It did not work well and I would not buy it again. It is too expensive."},
    {"role": "assistant", "content": "Negative"},
    {"role": "user", "content": "I do not know what to say; my son uses it, but he does not love it."},
    {"role": "assistant", "content": "Neutral"},
]
print(return_OAIResponse("I'm not going to return it, but I don't plan to buy it again.", context_user))


## Exercise

- Choose a task (for example, extracting fields, classifying feedback, or rewriting text).
    - For example: Extract the product name, price, and category from a product description.
- Create zero-shot, one-shot, and few-shot versions of your prompt.
- Test each version on the same set of at least five inputs.
- Compare correctness, consistency, and adherence to the requested format. Include any failures or hallucinations you observe.
- Write a one-page report summarizing your findings and what you learned.


## Exercise

Your task is to extract structured information from product descriptions.

Use the following product description as your starting example:

> The SoundWave Pro wireless headphones feature noise cancellation, Bluetooth 5.3 connectivity, and up to 30 hours of battery life. Available in black, they cost €89.99 and belong to the Electronics category.

1. Create zero-shot, one-shot, and few-shot prompts to extract the following fields:
   - **Product name**
   - **Price**
   - **Category**

2. Generate three additional product descriptions (using code or an LLM) and test all three prompts on the same descriptions.

3. Compare the results in terms of accuracy, consistency, and adherence to the expected output format. Include any errors or hallucinations you observe.

4. Write a brief, simple report summarizing your findings and what you learned about the impact of examples on model performance.

## Notes

- Model output can vary and may contain factual errors; verify factual answers independently.
- Remember to keep API keys private. API usage may incur charges.
